In [2]:
# ============================================================
# Summarize Griffin mean coverage, central coverage, and
# amplitude across all 22 target TFBSs.
#
# Output:
#   1. QC_Summary
#   2. All_TFBS_Long
#   3. One wide-format sheet per biological module
# ============================================================

from pathlib import Path

import pandas as pd
from openpyxl.styles import (
    Alignment,
    Border,
    Font,
    PatternFill,
    Side,
)
from openpyxl.utils import get_column_letter


# ============================================================
# 1. Paths
# ============================================================

PROJECT_DIR = Path(
    "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/"
    "Griffin/griffin_nucleosome_profiling"
)

RESULTS_DIR = PROJECT_DIR / "results"

OUTPUT_XLSX = (
    PROJECT_DIR /
    "Griffin_22TF_GC_corrected_summary_by_module.xlsx"
)


# ============================================================
# 2. TFBS modules
# ============================================================

MODULES = {
    "Interferon": [
        "IRF1",
        "STAT1",
        "STAT2",
    ],

    "HLA_Regulation": [
        "CREB1",
        "NFYA",
        "NFYC",
        "RFX1",
        "RFX5",
    ],

    "Breast_Lineage": [
        "ESR1",
        "FOXA1",
        "GATA3",
    ],

    "Bone_TGF_BMP": [
        "RUNX2",
        "SMAD1",
        "SMAD3",
        "SMAD4",
        "JUN",
    ],

    "Osteoclast_Inflammation": [
        "NFATC1",
        "SPI1",
        "RELA",
    ],

    "General_Tumor": [
        "MYC",
        "TP53",
    ],

    "Technical_Control": [
        "CTCF",
    ],
}


# Preserve the module and TF order defined above.
TARGET_SITES = [
    site
    for sites in MODULES.values()
    for site in sites
]

FEATURES = [
    "mean_coverage",
    "central_coverage",
    "amplitude",
]

COLUMNS_TO_READ = [
    "sample",
    "site_name",
    *FEATURES,
]


# Map every TFBS to its module.
SITE_TO_MODULE = {
    site: module
    for module, sites in MODULES.items()
    for site in sites
}


# ============================================================
# 3. Read all completed GC-corrected files
# ============================================================

all_results = []

missing_files = []
empty_files = []
failed_files = []
invalid_numeric_rows = 0

sample_dirs = sorted(
    folder
    for folder in RESULTS_DIR.iterdir()
    if folder.is_dir()
)


for sample_dir in sample_dirs:

    sample_name = sample_dir.name

    coverage_file = (
        sample_dir /
        f"{sample_name}.GC_corrected.coverage.tsv"
    )

    if not coverage_file.is_file():
        missing_files.append(sample_name)
        continue

    if coverage_file.stat().st_size == 0:
        empty_files.append(sample_name)
        continue

    try:
        df = pd.read_csv(
            coverage_file,
            sep="\t",
            usecols=COLUMNS_TO_READ,
        )

        # Convert site_STAT1 to STAT1, etc.
        df["site_name"] = (
            df["site_name"]
            .astype(str)
            .str.strip()
            .str.replace(r"^site_", "", regex=True)
        )

        # Retain only the 22 requested TFBSs.
        df = df[
            df["site_name"].isin(TARGET_SITES)
        ].copy()

        # Use the sample-folder name when the sample column is missing.
        df["sample"] = (
            df["sample"]
            .fillna(sample_name)
            .astype(str)
            .str.strip()
            .replace("", sample_name)
        )

        # Convert Griffin measurements to numeric.
        for feature in FEATURES:
            df[feature] = pd.to_numeric(
                df[feature],
                errors="coerce",
            )

        # Remove rows with missing/non-numeric Griffin measurements.
        invalid_mask = df[FEATURES].isna().any(axis=1)

        invalid_numeric_rows += int(
            invalid_mask.sum()
        )

        df = df.loc[~invalid_mask].copy()

        all_results.append(df)

    except Exception as error:
        failed_files.append(
            {
                "sample": sample_name,
                "error": str(error),
            }
        )


# ============================================================
# 4. Combine and validate results
# ============================================================

if not all_results:
    raise RuntimeError(
        "No valid GC-corrected Griffin results were found."
    )

combined_df = pd.concat(
    all_results,
    ignore_index=True,
)

combined_df = combined_df[
    [
        "sample",
        "site_name",
        "mean_coverage",
        "central_coverage",
        "amplitude",
    ]
].copy()


# Check duplicate sample-TFBS combinations.
duplicate_mask = combined_df.duplicated(
    subset=["sample", "site_name"],
    keep="first",
)

duplicate_count = int(
    duplicate_mask.sum()
)

if duplicate_count > 0:
    combined_df = combined_df.loc[
        ~duplicate_mask
    ].copy()


# Add module name.
combined_df["module"] = (
    combined_df["site_name"]
    .map(SITE_TO_MODULE)
)


# Sort using the predefined module and TF order.
module_order = list(MODULES.keys())

combined_df["module"] = pd.Categorical(
    combined_df["module"],
    categories=module_order,
    ordered=True,
)

combined_df["site_name"] = pd.Categorical(
    combined_df["site_name"],
    categories=TARGET_SITES,
    ordered=True,
)

combined_df = (
    combined_df
    .sort_values(
        [
            "module",
            "site_name",
            "sample",
        ]
    )
    .reset_index(drop=True)
)


# Samples successfully read.
completed_samples = sorted(
    combined_df["sample"]
    .dropna()
    .unique()
)

number_of_completed_samples = len(
    completed_samples
)


# Count samples available for each TFBS.
site_counts = (
    combined_df
    .groupby(
        "site_name",
        observed=False,
    )["sample"]
    .nunique()
    .reindex(
        TARGET_SITES,
        fill_value=0,
    )
)


# TFBSs completely absent from results.
missing_sites = [
    site
    for site in TARGET_SITES
    if site_counts.loc[site] == 0
]


# Expected sample-site combinations based on readable files.
expected_pairs = (
    len(all_results) *
    len(TARGET_SITES)
)

observed_pairs = len(combined_df)

missing_sample_site_pairs = (
    expected_pairs -
    observed_pairs
)


# ============================================================
# 5. Important console checks
# ============================================================

print(f"Sample folders found: {len(sample_dirs)}")
print(f"Coverage files read: {len(all_results)}")
print(f"Samples represented in output: {number_of_completed_samples}")
print(f"Missing coverage files: {len(missing_files)}")
print(f"Empty coverage files: {len(empty_files)}")
print(f"Failed files: {len(failed_files)}")
print(f"Invalid numeric rows removed: {invalid_numeric_rows}")
print(f"Duplicate sample-TFBS rows removed: {duplicate_count}")
print(f"Missing sample-TFBS combinations: {missing_sample_site_pairs}")

if missing_sites:
    print(
        "TFBSs with no results: "
        + ", ".join(missing_sites)
    )
else:
    print("All 22 target TFBSs were detected.")


# ============================================================
# 6. Prepare QC tables
# ============================================================

run_summary_df = pd.DataFrame(
    {
        "Metric": [
            "Sample folders found",
            "Coverage files read",
            "Samples represented in output",
            "Expected TFBSs",
            "Observed sample-TFBS rows",
            "Missing sample-TFBS combinations",
            "Missing coverage files",
            "Empty coverage files",
            "Failed files",
            "Invalid numeric rows removed",
            "Duplicate rows removed",
        ],
        "Value": [
            len(sample_dirs),
            len(all_results),
            number_of_completed_samples,
            len(TARGET_SITES),
            observed_pairs,
            missing_sample_site_pairs,
            len(missing_files),
            len(empty_files),
            len(failed_files),
            invalid_numeric_rows,
            duplicate_count,
        ],
    }
)


site_qc_df = pd.DataFrame(
    {
        "Module": [
            SITE_TO_MODULE[site]
            for site in TARGET_SITES
        ],
        "TFBS": TARGET_SITES,
        "Samples_with_result": [
            int(site_counts.loc[site])
            for site in TARGET_SITES
        ],
        "Missing_vs_files_read": [
            len(all_results) -
            int(site_counts.loc[site])
            for site in TARGET_SITES
        ],
    }
)


# Long-format table is best for GEE, ANCOVA, and plotting.
long_output_df = combined_df[
    [
        "sample",
        "module",
        "site_name",
        "mean_coverage",
        "central_coverage",
        "amplitude",
    ]
].copy()


# Convert categorical columns back to text for Excel.
long_output_df["module"] = (
    long_output_df["module"]
    .astype(str)
)

long_output_df["site_name"] = (
    long_output_df["site_name"]
    .astype(str)
)


# ============================================================
# 7. Excel formatting settings
# ============================================================

MODULE_COLORS = {
    "Interferon": "4472C4",
    "HLA_Regulation": "5B9BD5",
    "Breast_Lineage": "C55A11",
    "Bone_TGF_BMP": "70AD47",
    "Osteoclast_Inflammation": "A64D79",
    "General_Tumor": "8064A2",
    "Technical_Control": "7F7F7F",
}

HEADER_FONT = Font(
    bold=True,
    color="FFFFFF",
)

SUBHEADER_FONT = Font(
    bold=True,
    color="FFFFFF",
)

CENTER_ALIGNMENT = Alignment(
    horizontal="center",
    vertical="center",
)

THIN_BORDER = Border(
    bottom=Side(
        style="thin",
        color="BFBFBF",
    )
)


# ============================================================
# 8. Write Excel workbook
# ============================================================

with pd.ExcelWriter(
    OUTPUT_XLSX,
    engine="openpyxl",
) as writer:

    # --------------------------------------------------------
    # QC summary sheet
    # --------------------------------------------------------

    run_summary_df.to_excel(
        writer,
        sheet_name="QC_Summary",
        index=False,
        startrow=1,
    )

    site_qc_startrow = (
        len(run_summary_df) + 5
    )

    site_qc_df.to_excel(
        writer,
        sheet_name="QC_Summary",
        index=False,
        startrow=site_qc_startrow,
    )

    qc_ws = writer.sheets["QC_Summary"]

    qc_ws["A1"] = "Griffin 22-TFBS Summary QC"
    qc_ws["A1"].font = Font(
        bold=True,
        size=14,
        color="FFFFFF",
    )
    qc_ws["A1"].fill = PatternFill(
        "solid",
        fgColor="1F4E78",
    )

    qc_ws.merge_cells("A1:D1")

    qc_ws.freeze_panes = "A3"

    for cell in qc_ws[2]:
        cell.font = HEADER_FONT
        cell.fill = PatternFill(
            "solid",
            fgColor="1F4E78",
        )
        cell.alignment = CENTER_ALIGNMENT

    site_header_row = site_qc_startrow + 1

    for cell in qc_ws[site_header_row]:
        cell.font = HEADER_FONT
        cell.fill = PatternFill(
            "solid",
            fgColor="1F4E78",
        )
        cell.alignment = CENTER_ALIGNMENT

    qc_ws.column_dimensions["A"].width = 32
    qc_ws.column_dimensions["B"].width = 25
    qc_ws.column_dimensions["C"].width = 22
    qc_ws.column_dimensions["D"].width = 24

    # Add missing filenames when applicable.
    detail_row = (
        site_qc_startrow +
        len(site_qc_df) +
        4
    )

    if missing_files:
        qc_ws.cell(
            row=detail_row,
            column=1,
            value="Missing coverage files",
        ).font = Font(bold=True)

        for offset, sample in enumerate(
            missing_files,
            start=1,
        ):
            qc_ws.cell(
                row=detail_row + offset,
                column=1,
                value=sample,
            )

        detail_row += len(missing_files) + 2

    if empty_files:
        qc_ws.cell(
            row=detail_row,
            column=1,
            value="Empty coverage files",
        ).font = Font(bold=True)

        for offset, sample in enumerate(
            empty_files,
            start=1,
        ):
            qc_ws.cell(
                row=detail_row + offset,
                column=1,
                value=sample,
            )

        detail_row += len(empty_files) + 2

    if failed_files:
        qc_ws.cell(
            row=detail_row,
            column=1,
            value="Failed files",
        ).font = Font(bold=True)

        qc_ws.cell(
            row=detail_row + 1,
            column=1,
            value="sample",
        ).font = Font(bold=True)

        qc_ws.cell(
            row=detail_row + 1,
            column=2,
            value="error",
        ).font = Font(bold=True)

        for offset, record in enumerate(
            failed_files,
            start=2,
        ):
            qc_ws.cell(
                row=detail_row + offset,
                column=1,
                value=record["sample"],
            )

            qc_ws.cell(
                row=detail_row + offset,
                column=2,
                value=record["error"],
            )


    # --------------------------------------------------------
    # Long-format sheet
    # --------------------------------------------------------

    long_output_df.to_excel(
        writer,
        sheet_name="All_TFBS_Long",
        index=False,
        float_format="%.5f",
    )

    long_ws = writer.sheets["All_TFBS_Long"]

    long_ws.freeze_panes = "A2"
    long_ws.auto_filter.ref = long_ws.dimensions

    for cell in long_ws[1]:
        cell.font = HEADER_FONT
        cell.fill = PatternFill(
            "solid",
            fgColor="1F4E78",
        )
        cell.alignment = CENTER_ALIGNMENT

    long_ws.column_dimensions["A"].width = 18
    long_ws.column_dimensions["B"].width = 28
    long_ws.column_dimensions["C"].width = 14
    long_ws.column_dimensions["D"].width = 18
    long_ws.column_dimensions["E"].width = 20
    long_ws.column_dimensions["F"].width = 15


    # --------------------------------------------------------
    # One wide sheet per biological module
    # --------------------------------------------------------

    for module_name, module_sites in MODULES.items():

        ws = writer.book.create_sheet(
            title=module_name
        )

        module_color = MODULE_COLORS[
            module_name
        ]

        # First column contains sample name.
        ws.merge_cells(
            start_row=1,
            start_column=1,
            end_row=2,
            end_column=1,
        )

        sample_header = ws.cell(
            row=1,
            column=1,
            value="sample",
        )

        sample_header.font = HEADER_FONT
        sample_header.fill = PatternFill(
            "solid",
            fgColor=module_color,
        )
        sample_header.alignment = CENTER_ALIGNMENT

        current_column = 2

        # Write the two-level header.
        for site in module_sites:

            start_column = current_column
            end_column = current_column + 2

            ws.merge_cells(
                start_row=1,
                start_column=start_column,
                end_row=1,
                end_column=end_column,
            )

            site_header = ws.cell(
                row=1,
                column=start_column,
                value=site,
            )

            site_header.font = HEADER_FONT
            site_header.fill = PatternFill(
                "solid",
                fgColor=module_color,
            )
            site_header.alignment = CENTER_ALIGNMENT

            for feature_index, feature in enumerate(
                FEATURES
            ):

                column_number = (
                    start_column +
                    feature_index
                )

                feature_header = ws.cell(
                    row=2,
                    column=column_number,
                    value=f"{site}_{feature}",
                )

                feature_header.font = SUBHEADER_FONT
                feature_header.fill = PatternFill(
                    "solid",
                    fgColor=module_color,
                )
                feature_header.alignment = CENTER_ALIGNMENT

            current_column += 3

        # Create wide sample-level table.
        module_wide_df = pd.DataFrame(
            {
                "sample": completed_samples
            }
        )

        for site in module_sites:

            site_df = (
                combined_df[
                    combined_df["site_name"].astype(str)
                    == site
                ]
                .set_index("sample")
            )

            for feature in FEATURES:

                output_column = (
                    f"{site}_{feature}"
                )

                module_wide_df[
                    output_column
                ] = (
                    module_wide_df["sample"]
                    .map(site_df[feature])
                )

        # Write sample-level values beginning at row 3.
        for row_number, row_values in enumerate(
            module_wide_df.itertuples(
                index=False,
                name=None,
            ),
            start=3,
        ):

            for column_number, value in enumerate(
                row_values,
                start=1,
            ):

                cell = ws.cell(
                    row=row_number,
                    column=column_number,
                )

                if pd.isna(value):
                    cell.value = None
                else:
                    cell.value = value

                if column_number > 1:
                    cell.number_format = "0.00000"

                cell.border = THIN_BORDER

        last_column = (
            1 +
            3 * len(module_sites)
        )

        last_row = (
            len(module_wide_df) +
            2
        )

        # Freeze sample and header columns/rows.
        ws.freeze_panes = "B3"

        # Filter based on the unique second-row headers.
        ws.auto_filter.ref = (
            f"A2:"
            f"{get_column_letter(last_column)}"
            f"{last_row}"
        )

        # Column widths.
        ws.column_dimensions["A"].width = 18

        for column_number in range(
            2,
            last_column + 1,
        ):
            ws.column_dimensions[
                get_column_letter(column_number)
            ].width = 20

        # Header row heights.
        ws.row_dimensions[1].height = 24
        ws.row_dimensions[2].height = 38

        # Rotate long second-row headers.
        for column_number in range(
            2,
            last_column + 1,
        ):
            ws.cell(
                row=2,
                column=column_number,
            ).alignment = Alignment(
                horizontal="center",
                vertical="center",
                text_rotation=45,
                wrap_text=True,
            )


# ============================================================
# 9. Final output
# ============================================================

print(
    "\nExcel workbook created:\n"
    f"{OUTPUT_XLSX}"
)

Sample folders found: 769
Coverage files read: 769
Samples represented in output: 769
Missing coverage files: 0
Empty coverage files: 0
Failed files: 0
Invalid numeric rows removed: 0
Duplicate sample-TFBS rows removed: 0
Missing sample-TFBS combinations: 0
All 22 target TFBSs were detected.

Excel workbook created:
/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Griffin/griffin_nucleosome_profiling/Griffin_22TF_GC_corrected_summary_by_module.xlsx


In [4]:
"""
Update the existing Griffin 22-TF workbook to a 30-TF workbook
==============================================================

This script:

1. Reads all completed Griffin GC-corrected coverage files.
2. Uses the existing workbook as the starting template:

   Griffin_22TF_GC_corrected_summary_by_module.xlsx

3. Updates the module sheets using all currently configured TFBSs.
4. Adds the eight new Breast_Lineage TFBSs:
   PGR, GRHL2, STAT3, NFKB1, TFAP2C, XBP1, FOXM1, FOXO3.
5. Saves the updated workbook as:

   Griffin_30TF_GC_corrected_summary_by_module.xlsx

The original 22-TF workbook is not overwritten.

No backup workbook is created.
No All_TFBS_Long sheet is created or updated.
"""

from pathlib import Path

import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter


# ============================================================
# 1. Paths
# ============================================================

PROJECT_DIR = Path(
    "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/"
    "Griffin/griffin_nucleosome_profiling"
)

RESULTS_DIR = PROJECT_DIR / "results"

SOURCE_XLSX = (
    PROJECT_DIR
    / "Griffin_22TF_GC_corrected_summary_by_module.xlsx"
)

OUTPUT_XLSX = (
    PROJECT_DIR
    / "Griffin_30TF_GC_corrected_summary_by_module.xlsx"
)


# ============================================================
# 2. TFBS modules
# ============================================================

MODULES = {
    "Interferon": [
        "IRF1",
        "STAT1",
        "STAT2",
    ],

    "HLA_Regulation": [
        "CREB1",
        "NFYA",
        "NFYC",
        "RFX1",
        "RFX5",
    ],

    "Breast_Lineage": [
        "ESR1",
        "FOXA1",
        "GATA3",
        "PGR",
        "GRHL2",
        "STAT3",
        "NFKB1",
        "TFAP2C",
        "XBP1",
        "FOXM1",
        "FOXO3",
    ],

    "Bone_TGF_BMP": [
        "RUNX2",
        "SMAD1",
        "SMAD3",
        "SMAD4",
        "JUN",
    ],

    "Osteoclast_Inflammation": [
        "NFATC1",
        "SPI1",
        "RELA",
    ],

    "General_Tumor": [
        "MYC",
        "TP53",
    ],

    "Technical_Control": [
        "CTCF",
    ],
}


ORIGINAL_MODULES = {
    "Interferon": [
        "IRF1",
        "STAT1",
        "STAT2",
    ],

    "HLA_Regulation": [
        "CREB1",
        "NFYA",
        "NFYC",
        "RFX1",
        "RFX5",
    ],

    "Breast_Lineage": [
        "ESR1",
        "FOXA1",
        "GATA3",
    ],

    "Bone_TGF_BMP": [
        "RUNX2",
        "SMAD1",
        "SMAD3",
        "SMAD4",
        "JUN",
    ],

    "Osteoclast_Inflammation": [
        "NFATC1",
        "SPI1",
        "RELA",
    ],

    "General_Tumor": [
        "MYC",
        "TP53",
    ],

    "Technical_Control": [
        "CTCF",
    ],
}


TARGET_SITES = [
    site
    for sites in MODULES.values()
    for site in sites
]

FEATURES = [
    "mean_coverage",
    "central_coverage",
    "amplitude",
]

COLUMNS_TO_READ = [
    "sample",
    "site_name",
    *FEATURES,
]

SITE_TO_MODULE = {
    site: module
    for module, sites in MODULES.items()
    for site in sites
}


# ============================================================
# 3. Excel formatting
# ============================================================

MODULE_COLORS = {
    "Interferon": "4472C4",
    "HLA_Regulation": "5B9BD5",
    "Breast_Lineage": "C55A11",
    "Bone_TGF_BMP": "70AD47",
    "Osteoclast_Inflammation": "A64D79",
    "General_Tumor": "8064A2",
    "Technical_Control": "7F7F7F",
}

HEADER_FONT = Font(
    bold=True,
    color="FFFFFF",
)

CENTER_ALIGNMENT = Alignment(
    horizontal="center",
    vertical="center",
)

ROTATED_ALIGNMENT = Alignment(
    horizontal="center",
    vertical="center",
    text_rotation=45,
    wrap_text=True,
)

THIN_BORDER = Border(
    bottom=Side(
        style="thin",
        color="BFBFBF",
    )
)


# ============================================================
# 4. Validate input paths
# ============================================================

if not RESULTS_DIR.is_dir():
    raise FileNotFoundError(
        f"Results directory not found:\n{RESULTS_DIR}"
    )

if not SOURCE_XLSX.is_file():
    raise FileNotFoundError(
        f"Existing 22-TF workbook not found:\n{SOURCE_XLSX}"
    )


# ============================================================
# 5. Read all completed Griffin files
# ============================================================

all_results = []

missing_files = []
empty_files = []
failed_files = []
invalid_numeric_rows = 0

sample_dirs = sorted(
    folder
    for folder in RESULTS_DIR.iterdir()
    if folder.is_dir()
)

for sample_dir in sample_dirs:

    sample_name = sample_dir.name

    coverage_file = (
        sample_dir
        / f"{sample_name}.GC_corrected.coverage.tsv"
    )

    if not coverage_file.is_file():
        missing_files.append(sample_name)
        continue

    if coverage_file.stat().st_size == 0:
        empty_files.append(sample_name)
        continue

    try:
        df = pd.read_csv(
            coverage_file,
            sep="\t",
            usecols=COLUMNS_TO_READ,
        )

        # Convert site_ESR1 to ESR1.
        df["site_name"] = (
            df["site_name"]
            .astype(str)
            .str.strip()
            .str.replace(
                r"^site_",
                "",
                regex=True,
            )
        )

        # Retain only the 30 configured TFBSs.
        df = df[
            df["site_name"].isin(TARGET_SITES)
        ].copy()

        if df.empty:
            continue

        # Use the sample-folder name if sample is missing.
        df["sample"] = (
            df["sample"]
            .fillna(sample_name)
            .astype(str)
            .str.strip()
            .replace("", sample_name)
        )

        for feature in FEATURES:
            df[feature] = pd.to_numeric(
                df[feature],
                errors="coerce",
            )

        invalid_mask = (
            df[FEATURES]
            .isna()
            .any(axis=1)
        )

        invalid_numeric_rows += int(
            invalid_mask.sum()
        )

        df = df.loc[
            ~invalid_mask
        ].copy()

        if not df.empty:
            all_results.append(df)

    except Exception as error:
        failed_files.append(
            {
                "sample": sample_name,
                "error": str(error),
            }
        )


if not all_results:
    raise RuntimeError(
        "No valid GC-corrected Griffin results were found."
    )


# ============================================================
# 6. Combine and validate results
# ============================================================

combined_df = pd.concat(
    all_results,
    ignore_index=True,
)

combined_df = combined_df[
    [
        "sample",
        "site_name",
        *FEATURES,
    ]
].copy()

combined_df["sample"] = (
    combined_df["sample"]
    .astype(str)
    .str.strip()
)

combined_df["site_name"] = (
    combined_df["site_name"]
    .astype(str)
    .str.strip()
)

duplicate_mask = combined_df.duplicated(
    subset=[
        "sample",
        "site_name",
    ],
    keep="last",
)

duplicate_count = int(
    duplicate_mask.sum()
)

combined_df = (
    combined_df
    .drop_duplicates(
        subset=[
            "sample",
            "site_name",
        ],
        keep="last",
    )
    .reset_index(drop=True)
)

combined_df["module"] = (
    combined_df["site_name"]
    .map(SITE_TO_MODULE)
)

completed_samples = sorted(
    combined_df["sample"]
    .dropna()
    .unique()
)

# These counts describe only the newly read result files.
# Final QC counts are recalculated later from the merged workbook sheets.
new_result_site_counts = (
    combined_df
    .groupby("site_name")["sample"]
    .nunique()
    .reindex(
        TARGET_SITES,
        fill_value=0,
    )
)


# ============================================================
# 7. Load existing 22-TF workbook
# ============================================================

workbook = load_workbook(
    SOURCE_XLSX
)

required_module_sheets = list(
    MODULES.keys()
)

missing_sheets = [
    sheet
    for sheet in required_module_sheets
    if sheet not in workbook.sheetnames
]

if missing_sheets:
    raise RuntimeError(
        "The source workbook is missing module sheets:\n"
        + "\n".join(missing_sheets)
    )

# The updated workbook does not need All_TFBS_Long.
if "All_TFBS_Long" in workbook.sheetnames:
    del workbook["All_TFBS_Long"]


# ============================================================
# 8. Helper functions
# ============================================================

def get_sample_rows(worksheet):
    """Map sample names in column A to worksheet row numbers."""

    sample_rows = {}

    for row_number in range(
        3,
        worksheet.max_row + 1,
    ):
        value = worksheet.cell(
            row=row_number,
            column=1,
        ).value

        if value is not None:
            sample_rows[
                str(value).strip()
            ] = row_number

    return sample_rows


def get_feature_columns(worksheet):
    """Map row-2 feature headers to worksheet column numbers."""

    feature_columns = {}

    for column_number in range(
        2,
        worksheet.max_column + 1,
    ):
        value = worksheet.cell(
            row=2,
            column=column_number,
        ).value

        if value is not None:
            feature_columns[
                str(value).strip()
            ] = column_number

    return feature_columns


def add_tfbs_block(
    worksheet,
    module_name,
    site,
    feature_columns,
):
    """Add three columns for one new TFBS."""

    module_color = MODULE_COLORS[
        module_name
    ]

    start_column = (
        worksheet.max_column + 1
    )

    end_column = (
        start_column
        + len(FEATURES)
        - 1
    )

    worksheet.merge_cells(
        start_row=1,
        start_column=start_column,
        end_row=1,
        end_column=end_column,
    )

    site_header = worksheet.cell(
        row=1,
        column=start_column,
        value=site,
    )

    site_header.font = HEADER_FONT
    site_header.fill = PatternFill(
        "solid",
        fgColor=module_color,
    )
    site_header.alignment = CENTER_ALIGNMENT

    for offset, feature in enumerate(
        FEATURES
    ):
        column_number = (
            start_column + offset
        )

        header_name = (
            f"{site}_{feature}"
        )

        header_cell = worksheet.cell(
            row=2,
            column=column_number,
            value=header_name,
        )

        header_cell.font = HEADER_FONT
        header_cell.fill = PatternFill(
            "solid",
            fgColor=module_color,
        )
        header_cell.alignment = (
            ROTATED_ALIGNMENT
        )

        worksheet.column_dimensions[
            get_column_letter(column_number)
        ].width = 20

        feature_columns[
            header_name
        ] = column_number


def update_module_sheet(
    worksheet,
    module_name,
    module_results,
):
    """Update one existing module worksheet."""

    sample_rows = get_sample_rows(
        worksheet
    )

    feature_columns = get_feature_columns(
        worksheet
    )

    added_tfbs = 0
    added_samples = 0
    updated_values = 0

    # Add TFBS blocks that are absent from the 22-TF workbook.
    for site in MODULES[module_name]:

        expected_headers = [
            f"{site}_{feature}"
            for feature in FEATURES
        ]

        if not all(
            header in feature_columns
            for header in expected_headers
        ):
            add_tfbs_block(
                worksheet=worksheet,
                module_name=module_name,
                site=site,
                feature_columns=feature_columns,
            )

            added_tfbs += 1

    # Update or add samples.
    for record in module_results.itertuples(
        index=False
    ):

        sample_name = str(
            record.sample
        ).strip()

        site = str(
            record.site_name
        ).strip()

        if sample_name not in sample_rows:

            row_number = (
                worksheet.max_row + 1
            )

            worksheet.cell(
                row=row_number,
                column=1,
                value=sample_name,
            )

            sample_rows[
                sample_name
            ] = row_number

            added_samples += 1

        row_number = (
            sample_rows[sample_name]
        )

        for feature in FEATURES:

            header_name = (
                f"{site}_{feature}"
            )

            column_number = (
                feature_columns[header_name]
            )

            value = getattr(
                record,
                feature,
            )

            cell = worksheet.cell(
                row=row_number,
                column=column_number,
                value=float(value),
            )

            cell.number_format = "0.00000"
            cell.border = THIN_BORDER

            updated_values += 1

    worksheet.freeze_panes = "B3"

    worksheet.auto_filter.ref = (
        f"A2:"
        f"{get_column_letter(worksheet.max_column)}"
        f"{worksheet.max_row}"
    )

    worksheet.row_dimensions[1].height = 24
    worksheet.row_dimensions[2].height = 40

    return {
        "added_tfbs": added_tfbs,
        "added_samples": added_samples,
        "updated_values": updated_values,
    }


# ============================================================
# 9. Update all module worksheets
# ============================================================

module_summary = []

for module_name in MODULES:

    module_results = combined_df[
        combined_df["module"]
        == module_name
    ].copy()

    result = update_module_sheet(
        worksheet=workbook[module_name],
        module_name=module_name,
        module_results=module_results,
    )

    existing_tfbs = len(
        ORIGINAL_MODULES.get(
            module_name,
            [],
        )
    )

    final_tfbs = len(
        MODULES[module_name]
    )

    module_summary.append(
        {
            "Module": module_name,
            "Existing_TFBS": existing_tfbs,
            "TFBS_added": (
                final_tfbs
                - existing_tfbs
            ),
            "Samples_added": result[
                "added_samples"
            ],
        }
    )


# ============================================================
# 10. Recalculate QC counts from merged module sheets
# ============================================================

# Count all samples represented anywhere in the merged module sheets.
merged_sample_names = set()

for module_name in MODULES:

    ws = workbook[module_name]

    for row_number in range(
        3,
        ws.max_row + 1,
    ):
        sample_value = ws.cell(
            row=row_number,
            column=1,
        ).value

        if sample_value is not None:
            sample_name = str(
                sample_value
            ).strip()

            if sample_name:
                merged_sample_names.add(
                    sample_name
                )


number_of_merged_samples = len(
    merged_sample_names
)


# Count non-missing values for each TFBS from the final merged worksheets.
# A sample is counted once for a TFBS when at least one of its three
# Griffin measurements is present.
merged_site_counts = {}

for module_name, module_sites in MODULES.items():

    ws = workbook[module_name]

    feature_columns = get_feature_columns(
        ws
    )

    for site in module_sites:

        site_feature_columns = [
            feature_columns.get(
                f"{site}_{feature}"
            )
            for feature in FEATURES
        ]

        site_feature_columns = [
            column_number
            for column_number in site_feature_columns
            if column_number is not None
        ]

        samples_with_result = set()

        if site_feature_columns:

            for row_number in range(
                3,
                ws.max_row + 1,
            ):

                sample_value = ws.cell(
                    row=row_number,
                    column=1,
                ).value

                if sample_value is None:
                    continue

                sample_name = str(
                    sample_value
                ).strip()

                if not sample_name:
                    continue

                has_result = any(
                    ws.cell(
                        row=row_number,
                        column=column_number,
                    ).value
                    not in (
                        None,
                        "",
                    )
                    for column_number in site_feature_columns
                )

                if has_result:
                    samples_with_result.add(
                        sample_name
                    )

        merged_site_counts[site] = len(
            samples_with_result
        )


missing_sites = [
    site
    for site in TARGET_SITES
    if merged_site_counts.get(
        site,
        0,
    ) == 0
]


# ============================================================
# 11. Update QC_Summary
# ============================================================

if "QC_Summary" in workbook.sheetnames:

    qc_ws = workbook[
        "QC_Summary"
    ]

    # Clear the existing QC sheet and write a concise updated summary.
    qc_ws.delete_rows(
        1,
        qc_ws.max_row,
    )

else:
    qc_ws = workbook.create_sheet(
        "QC_Summary",
        0,
    )


qc_ws["A1"] = (
    "Griffin 30-TFBS Summary QC"
)

qc_ws["A1"].font = Font(
    bold=True,
    size=14,
    color="FFFFFF",
)

qc_ws["A1"].fill = PatternFill(
    "solid",
    fgColor="1F4E78",
)

qc_ws.merge_cells("A1:D1")


qc_metrics = [
    ("Metric", "Value"),
    ("Sample folders found", len(sample_dirs)),
    ("Coverage files read", len(all_results)),
    ("Samples represented", number_of_merged_samples),
    ("Configured TFBSs", len(TARGET_SITES)),
    ("Missing coverage files", len(missing_files)),
    ("Empty coverage files", len(empty_files)),
    ("Failed files", len(failed_files)),
    ("Invalid numeric rows removed", invalid_numeric_rows),
    ("Duplicate sample-TFBS rows removed", duplicate_count),
]

for row_number, (
    metric,
    value,
) in enumerate(
    qc_metrics,
    start=2,
):
    qc_ws.cell(
        row=row_number,
        column=1,
        value=metric,
    )

    qc_ws.cell(
        row=row_number,
        column=2,
        value=value,
    )


for cell in qc_ws[2][:2]:
    cell.font = HEADER_FONT
    cell.fill = PatternFill(
        "solid",
        fgColor="1F4E78",
    )
    cell.alignment = CENTER_ALIGNMENT


site_qc_start_row = (
    len(qc_metrics) + 4
)

site_headers = [
    "Module",
    "TFBS",
    "Samples_with_result",
    "Missing_vs_completed_samples",
]

for column_number, header in enumerate(
    site_headers,
    start=1,
):
    cell = qc_ws.cell(
        row=site_qc_start_row,
        column=column_number,
        value=header,
    )

    cell.font = HEADER_FONT
    cell.fill = PatternFill(
        "solid",
        fgColor="1F4E78",
    )
    cell.alignment = CENTER_ALIGNMENT


for row_offset, site in enumerate(
    TARGET_SITES,
    start=1,
):

    values = [
        SITE_TO_MODULE[site],
        site,
        int(merged_site_counts.get(site, 0)),
        number_of_merged_samples
        - int(merged_site_counts.get(site, 0)),
    ]

    for column_number, value in enumerate(
        values,
        start=1,
    ):
        qc_ws.cell(
            row=site_qc_start_row + row_offset,
            column=column_number,
            value=value,
        )


qc_ws.freeze_panes = "A3"
qc_ws.column_dimensions["A"].width = 34
qc_ws.column_dimensions["B"].width = 25
qc_ws.column_dimensions["C"].width = 24
qc_ws.column_dimensions["D"].width = 30


# ============================================================
# 12. Save updated 30-TF workbook
# ============================================================

workbook.save(
    OUTPUT_XLSX
)


# ============================================================
# 13. Final console report
# ============================================================

print(f"Sample folders found: {len(sample_dirs)}")
print(f"Coverage files read: {len(all_results)}")
print(f"Samples represented in merged workbook: {number_of_merged_samples}")
print(f"Configured TFBSs: {len(TARGET_SITES)}")
print(f"Missing coverage files: {len(missing_files)}")
print(f"Empty coverage files: {len(empty_files)}")
print(f"Failed files: {len(failed_files)}")
print(f"Invalid numeric rows removed: {invalid_numeric_rows}")
print(f"Duplicate sample-TFBS rows removed: {duplicate_count}")

if missing_sites:
    print(
        "TFBSs with no results: "
        + ", ".join(missing_sites)
    )
else:
    print(
        "All 30 configured TFBSs were detected."
    )

print(
    "\nUpdated workbook created:\n"
    f"{OUTPUT_XLSX}"
)

print(
    "\nModule update summary:"
)

print(
    pd.DataFrame(
        module_summary
    ).to_string(
        index=False
    )
)

Sample folders found: 769
Coverage files read: 769
Samples represented in merged workbook: 769
Configured TFBSs: 30
Missing coverage files: 0
Empty coverage files: 0
Failed files: 0
Invalid numeric rows removed: 0
Duplicate sample-TFBS rows removed: 0
All 30 configured TFBSs were detected.

Updated workbook created:
/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Griffin/griffin_nucleosome_profiling/Griffin_30TF_GC_corrected_summary_by_module.xlsx

Module update summary:
                 Module  Existing_TFBS  TFBS_added  Samples_added
             Interferon              3           0              0
         HLA_Regulation              5           0              0
         Breast_Lineage              3           8              0
           Bone_TGF_BMP              5           0              0
Osteoclast_Inflammation              3           0              0
          General_Tumor              2           0              0
      Technical_Control              1           0       

In [1]:
"""
Update the existing Griffin 30-TF workbook to a 44-TF workbook
==============================================================

This script:

1. Reads all completed Griffin GC-corrected coverage files.
2. Uses the existing workbook as the starting template:

   Griffin_30TF_GC_corrected_summary_by_module.xlsx

3. Updates the module sheets using all currently configured TFBSs.
4. Adds the fourteen new Breast_Lineage TFBSs:
   AR, ESR2, FOXO1, RUNX1, E2F1, E2F7, TEAD1, TEAD4,
   FOSL1, BACH1, HIF1A, ETS1, CEBPB, ELF3.
5. Saves the updated workbook as:

   Griffin_44TF_GC_corrected_summary_by_module.xlsx

The original 30-TF workbook is not overwritten.

No backup workbook is created.
No All_TFBS_Long sheet is created or updated.
"""

from pathlib import Path

import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter


# ============================================================
# 1. Paths
# ============================================================

PROJECT_DIR = Path(
    "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/"
    "Griffin/griffin_nucleosome_profiling"
)

RESULTS_DIR = PROJECT_DIR / "results"

SOURCE_XLSX = (
    PROJECT_DIR
    / "Griffin_30TF_GC_corrected_summary_by_module.xlsx"
)

OUTPUT_XLSX = (
    PROJECT_DIR
    / "Griffin_44TF_GC_corrected_summary_by_module.xlsx"
)


# ============================================================
# 2. TFBS modules
# ============================================================

MODULES = {
    "Interferon": [
        "IRF1",
        "STAT1",
        "STAT2",
    ],

    "HLA_Regulation": [
        "CREB1",
        "NFYA",
        "NFYC",
        "RFX1",
        "RFX5",
    ],

    "Breast_Lineage": [
        "ESR1",
        "FOXA1",
        "GATA3",
        "PGR",
        "GRHL2",
        "STAT3",
        "NFKB1",
        "TFAP2C",
        "XBP1",
        "FOXM1",
        "FOXO3",
        "AR",
        "ESR2",
        "FOXO1",
        "RUNX1",
        "E2F1",
        "E2F7",
        "TEAD1",
        "TEAD4",
        "FOSL1",
        "BACH1",
        "HIF1A",
        "ETS1",
        "CEBPB",
        "ELF3",
    ],

    "Bone_TGF_BMP": [
        "RUNX2",
        "SMAD1",
        "SMAD3",
        "SMAD4",
        "JUN",
    ],

    "Osteoclast_Inflammation": [
        "NFATC1",
        "SPI1",
        "RELA",
    ],

    "General_Tumor": [
        "MYC",
        "TP53",
    ],

    "Technical_Control": [
        "CTCF",
    ],
}


ORIGINAL_MODULES = {
    "Interferon": [
        "IRF1",
        "STAT1",
        "STAT2",
    ],

    "HLA_Regulation": [
        "CREB1",
        "NFYA",
        "NFYC",
        "RFX1",
        "RFX5",
    ],

    "Breast_Lineage": [
        "ESR1",
        "FOXA1",
        "GATA3",
        "PGR",
        "GRHL2",
        "STAT3",
        "NFKB1",
        "TFAP2C",
        "XBP1",
        "FOXM1",
        "FOXO3",
    ],

    "Bone_TGF_BMP": [
        "RUNX2",
        "SMAD1",
        "SMAD3",
        "SMAD4",
        "JUN",
    ],

    "Osteoclast_Inflammation": [
        "NFATC1",
        "SPI1",
        "RELA",
    ],

    "General_Tumor": [
        "MYC",
        "TP53",
    ],

    "Technical_Control": [
        "CTCF",
    ],
}


TARGET_SITES = [
    site
    for sites in MODULES.values()
    for site in sites
]

FEATURES = [
    "mean_coverage",
    "central_coverage",
    "amplitude",
]

COLUMNS_TO_READ = [
    "sample",
    "site_name",
    *FEATURES,
]

SITE_TO_MODULE = {
    site: module
    for module, sites in MODULES.items()
    for site in sites
}


# ============================================================
# 3. Excel formatting
# ============================================================

MODULE_COLORS = {
    "Interferon": "4472C4",
    "HLA_Regulation": "5B9BD5",
    "Breast_Lineage": "C55A11",
    "Bone_TGF_BMP": "70AD47",
    "Osteoclast_Inflammation": "A64D79",
    "General_Tumor": "8064A2",
    "Technical_Control": "7F7F7F",
}

HEADER_FONT = Font(
    bold=True,
    color="FFFFFF",
)

CENTER_ALIGNMENT = Alignment(
    horizontal="center",
    vertical="center",
)

ROTATED_ALIGNMENT = Alignment(
    horizontal="center",
    vertical="center",
    text_rotation=45,
    wrap_text=True,
)

THIN_BORDER = Border(
    bottom=Side(
        style="thin",
        color="BFBFBF",
    )
)


# ============================================================
# 4. Validate input paths
# ============================================================

if not RESULTS_DIR.is_dir():
    raise FileNotFoundError(
        f"Results directory not found:\n{RESULTS_DIR}"
    )

if not SOURCE_XLSX.is_file():
    raise FileNotFoundError(
        f"Existing 30-TF workbook not found:\n{SOURCE_XLSX}"
    )


# ============================================================
# 5. Read all completed Griffin files
# ============================================================

all_results = []

missing_files = []
empty_files = []
failed_files = []
invalid_numeric_rows = 0

sample_dirs = sorted(
    folder
    for folder in RESULTS_DIR.iterdir()
    if folder.is_dir()
)

for sample_dir in sample_dirs:

    sample_name = sample_dir.name

    coverage_file = (
        sample_dir
        / f"{sample_name}.GC_corrected.coverage.tsv"
    )

    if not coverage_file.is_file():
        missing_files.append(sample_name)
        continue

    if coverage_file.stat().st_size == 0:
        empty_files.append(sample_name)
        continue

    try:
        df = pd.read_csv(
            coverage_file,
            sep="\t",
            usecols=COLUMNS_TO_READ,
        )

        # Convert site_ESR1 to ESR1.
        df["site_name"] = (
            df["site_name"]
            .astype(str)
            .str.strip()
            .str.replace(
                r"^site_",
                "",
                regex=True,
            )
        )

        # Retain only the 44 configured TFBSs.
        df = df[
            df["site_name"].isin(TARGET_SITES)
        ].copy()

        if df.empty:
            continue

        # Use the sample-folder name if sample is missing.
        df["sample"] = (
            df["sample"]
            .fillna(sample_name)
            .astype(str)
            .str.strip()
            .replace("", sample_name)
        )

        for feature in FEATURES:
            df[feature] = pd.to_numeric(
                df[feature],
                errors="coerce",
            )

        invalid_mask = (
            df[FEATURES]
            .isna()
            .any(axis=1)
        )

        invalid_numeric_rows += int(
            invalid_mask.sum()
        )

        df = df.loc[
            ~invalid_mask
        ].copy()

        if not df.empty:
            all_results.append(df)

    except Exception as error:
        failed_files.append(
            {
                "sample": sample_name,
                "error": str(error),
            }
        )


if not all_results:
    raise RuntimeError(
        "No valid GC-corrected Griffin results were found."
    )


# ============================================================
# 6. Combine and validate results
# ============================================================

combined_df = pd.concat(
    all_results,
    ignore_index=True,
)

combined_df = combined_df[
    [
        "sample",
        "site_name",
        *FEATURES,
    ]
].copy()

combined_df["sample"] = (
    combined_df["sample"]
    .astype(str)
    .str.strip()
)

combined_df["site_name"] = (
    combined_df["site_name"]
    .astype(str)
    .str.strip()
)

duplicate_mask = combined_df.duplicated(
    subset=[
        "sample",
        "site_name",
    ],
    keep="last",
)

duplicate_count = int(
    duplicate_mask.sum()
)

combined_df = (
    combined_df
    .drop_duplicates(
        subset=[
            "sample",
            "site_name",
        ],
        keep="last",
    )
    .reset_index(drop=True)
)

combined_df["module"] = (
    combined_df["site_name"]
    .map(SITE_TO_MODULE)
)

completed_samples = sorted(
    combined_df["sample"]
    .dropna()
    .unique()
)

# These counts describe only the newly read result files.
# Final QC counts are recalculated later from the merged workbook sheets.
new_result_site_counts = (
    combined_df
    .groupby("site_name")["sample"]
    .nunique()
    .reindex(
        TARGET_SITES,
        fill_value=0,
    )
)


# ============================================================
# 7. Load existing 30-TF workbook
# ============================================================

workbook = load_workbook(
    SOURCE_XLSX
)

required_module_sheets = list(
    MODULES.keys()
)

missing_sheets = [
    sheet
    for sheet in required_module_sheets
    if sheet not in workbook.sheetnames
]

if missing_sheets:
    raise RuntimeError(
        "The source workbook is missing module sheets:\n"
        + "\n".join(missing_sheets)
    )

# The updated workbook does not need All_TFBS_Long.
if "All_TFBS_Long" in workbook.sheetnames:
    del workbook["All_TFBS_Long"]


# ============================================================
# 8. Helper functions
# ============================================================

def get_sample_rows(worksheet):
    """Map sample names in column A to worksheet row numbers."""

    sample_rows = {}

    for row_number in range(
        3,
        worksheet.max_row + 1,
    ):
        value = worksheet.cell(
            row=row_number,
            column=1,
        ).value

        if value is not None:
            sample_rows[
                str(value).strip()
            ] = row_number

    return sample_rows


def get_feature_columns(worksheet):
    """Map row-2 feature headers to worksheet column numbers."""

    feature_columns = {}

    for column_number in range(
        2,
        worksheet.max_column + 1,
    ):
        value = worksheet.cell(
            row=2,
            column=column_number,
        ).value

        if value is not None:
            feature_columns[
                str(value).strip()
            ] = column_number

    return feature_columns


def add_tfbs_block(
    worksheet,
    module_name,
    site,
    feature_columns,
):
    """Add three columns for one new TFBS."""

    module_color = MODULE_COLORS[
        module_name
    ]

    start_column = (
        worksheet.max_column + 1
    )

    end_column = (
        start_column
        + len(FEATURES)
        - 1
    )

    worksheet.merge_cells(
        start_row=1,
        start_column=start_column,
        end_row=1,
        end_column=end_column,
    )

    site_header = worksheet.cell(
        row=1,
        column=start_column,
        value=site,
    )

    site_header.font = HEADER_FONT
    site_header.fill = PatternFill(
        "solid",
        fgColor=module_color,
    )
    site_header.alignment = CENTER_ALIGNMENT

    for offset, feature in enumerate(
        FEATURES
    ):
        column_number = (
            start_column + offset
        )

        header_name = (
            f"{site}_{feature}"
        )

        header_cell = worksheet.cell(
            row=2,
            column=column_number,
            value=header_name,
        )

        header_cell.font = HEADER_FONT
        header_cell.fill = PatternFill(
            "solid",
            fgColor=module_color,
        )
        header_cell.alignment = (
            ROTATED_ALIGNMENT
        )

        worksheet.column_dimensions[
            get_column_letter(column_number)
        ].width = 20

        feature_columns[
            header_name
        ] = column_number


def update_module_sheet(
    worksheet,
    module_name,
    module_results,
):
    """Update one existing module worksheet."""

    sample_rows = get_sample_rows(
        worksheet
    )

    feature_columns = get_feature_columns(
        worksheet
    )

    added_tfbs = 0
    added_samples = 0
    updated_values = 0

    # Add TFBS blocks that are absent from the 30-TF workbook.
    for site in MODULES[module_name]:

        expected_headers = [
            f"{site}_{feature}"
            for feature in FEATURES
        ]

        if not all(
            header in feature_columns
            for header in expected_headers
        ):
            add_tfbs_block(
                worksheet=worksheet,
                module_name=module_name,
                site=site,
                feature_columns=feature_columns,
            )

            added_tfbs += 1

    # Update or add samples.
    for record in module_results.itertuples(
        index=False
    ):

        sample_name = str(
            record.sample
        ).strip()

        site = str(
            record.site_name
        ).strip()

        if sample_name not in sample_rows:

            row_number = (
                worksheet.max_row + 1
            )

            worksheet.cell(
                row=row_number,
                column=1,
                value=sample_name,
            )

            sample_rows[
                sample_name
            ] = row_number

            added_samples += 1

        row_number = (
            sample_rows[sample_name]
        )

        for feature in FEATURES:

            header_name = (
                f"{site}_{feature}"
            )

            column_number = (
                feature_columns[header_name]
            )

            value = getattr(
                record,
                feature,
            )

            cell = worksheet.cell(
                row=row_number,
                column=column_number,
                value=float(value),
            )

            cell.number_format = "0.00000"
            cell.border = THIN_BORDER

            updated_values += 1

    worksheet.freeze_panes = "B3"

    worksheet.auto_filter.ref = (
        f"A2:"
        f"{get_column_letter(worksheet.max_column)}"
        f"{worksheet.max_row}"
    )

    worksheet.row_dimensions[1].height = 24
    worksheet.row_dimensions[2].height = 40

    return {
        "added_tfbs": added_tfbs,
        "added_samples": added_samples,
        "updated_values": updated_values,
    }


# ============================================================
# 9. Update all module worksheets
# ============================================================

module_summary = []

for module_name in MODULES:

    module_results = combined_df[
        combined_df["module"]
        == module_name
    ].copy()

    result = update_module_sheet(
        worksheet=workbook[module_name],
        module_name=module_name,
        module_results=module_results,
    )

    existing_tfbs = len(
        ORIGINAL_MODULES.get(
            module_name,
            [],
        )
    )

    final_tfbs = len(
        MODULES[module_name]
    )

    module_summary.append(
        {
            "Module": module_name,
            "Existing_TFBS": existing_tfbs,
            "TFBS_added": (
                final_tfbs
                - existing_tfbs
            ),
            "Samples_added": result[
                "added_samples"
            ],
        }
    )


# ============================================================
# 10. Recalculate QC counts from merged module sheets
# ============================================================

# Count all samples represented anywhere in the merged module sheets.
merged_sample_names = set()

for module_name in MODULES:

    ws = workbook[module_name]

    for row_number in range(
        3,
        ws.max_row + 1,
    ):
        sample_value = ws.cell(
            row=row_number,
            column=1,
        ).value

        if sample_value is not None:
            sample_name = str(
                sample_value
            ).strip()

            if sample_name:
                merged_sample_names.add(
                    sample_name
                )


number_of_merged_samples = len(
    merged_sample_names
)


# Count non-missing values for each TFBS from the final merged worksheets.
# A sample is counted once for a TFBS when at least one of its three
# Griffin measurements is present.
merged_site_counts = {}

for module_name, module_sites in MODULES.items():

    ws = workbook[module_name]

    feature_columns = get_feature_columns(
        ws
    )

    for site in module_sites:

        site_feature_columns = [
            feature_columns.get(
                f"{site}_{feature}"
            )
            for feature in FEATURES
        ]

        site_feature_columns = [
            column_number
            for column_number in site_feature_columns
            if column_number is not None
        ]

        samples_with_result = set()

        if site_feature_columns:

            for row_number in range(
                3,
                ws.max_row + 1,
            ):

                sample_value = ws.cell(
                    row=row_number,
                    column=1,
                ).value

                if sample_value is None:
                    continue

                sample_name = str(
                    sample_value
                ).strip()

                if not sample_name:
                    continue

                has_result = any(
                    ws.cell(
                        row=row_number,
                        column=column_number,
                    ).value
                    not in (
                        None,
                        "",
                    )
                    for column_number in site_feature_columns
                )

                if has_result:
                    samples_with_result.add(
                        sample_name
                    )

        merged_site_counts[site] = len(
            samples_with_result
        )


missing_sites = [
    site
    for site in TARGET_SITES
    if merged_site_counts.get(
        site,
        0,
    ) == 0
]


# ============================================================
# 11. Update QC_Summary
# ============================================================

if "QC_Summary" in workbook.sheetnames:

    qc_ws = workbook[
        "QC_Summary"
    ]

    # Clear the existing QC sheet and write a concise updated summary.
    qc_ws.delete_rows(
        1,
        qc_ws.max_row,
    )

else:
    qc_ws = workbook.create_sheet(
        "QC_Summary",
        0,
    )


qc_ws["A1"] = (
    "Griffin 44-TFBS Summary QC"
)

qc_ws["A1"].font = Font(
    bold=True,
    size=14,
    color="FFFFFF",
)

qc_ws["A1"].fill = PatternFill(
    "solid",
    fgColor="1F4E78",
)

qc_ws.merge_cells("A1:D1")


qc_metrics = [
    ("Metric", "Value"),
    ("Sample folders found", len(sample_dirs)),
    ("Coverage files read", len(all_results)),
    ("Samples represented", number_of_merged_samples),
    ("Configured TFBSs", len(TARGET_SITES)),
    ("Missing coverage files", len(missing_files)),
    ("Empty coverage files", len(empty_files)),
    ("Failed files", len(failed_files)),
    ("Invalid numeric rows removed", invalid_numeric_rows),
    ("Duplicate sample-TFBS rows removed", duplicate_count),
]

for row_number, (
    metric,
    value,
) in enumerate(
    qc_metrics,
    start=2,
):
    qc_ws.cell(
        row=row_number,
        column=1,
        value=metric,
    )

    qc_ws.cell(
        row=row_number,
        column=2,
        value=value,
    )


for cell in qc_ws[2][:2]:
    cell.font = HEADER_FONT
    cell.fill = PatternFill(
        "solid",
        fgColor="1F4E78",
    )
    cell.alignment = CENTER_ALIGNMENT


site_qc_start_row = (
    len(qc_metrics) + 4
)

site_headers = [
    "Module",
    "TFBS",
    "Samples_with_result",
    "Missing_vs_completed_samples",
]

for column_number, header in enumerate(
    site_headers,
    start=1,
):
    cell = qc_ws.cell(
        row=site_qc_start_row,
        column=column_number,
        value=header,
    )

    cell.font = HEADER_FONT
    cell.fill = PatternFill(
        "solid",
        fgColor="1F4E78",
    )
    cell.alignment = CENTER_ALIGNMENT


for row_offset, site in enumerate(
    TARGET_SITES,
    start=1,
):

    values = [
        SITE_TO_MODULE[site],
        site,
        int(merged_site_counts.get(site, 0)),
        number_of_merged_samples
        - int(merged_site_counts.get(site, 0)),
    ]

    for column_number, value in enumerate(
        values,
        start=1,
    ):
        qc_ws.cell(
            row=site_qc_start_row + row_offset,
            column=column_number,
            value=value,
        )


qc_ws.freeze_panes = "A3"
qc_ws.column_dimensions["A"].width = 34
qc_ws.column_dimensions["B"].width = 25
qc_ws.column_dimensions["C"].width = 24
qc_ws.column_dimensions["D"].width = 30


# ============================================================
# 12. Save updated 44-TF workbook
# ============================================================

workbook.save(
    OUTPUT_XLSX
)


# ============================================================
# 13. Final console report
# ============================================================

print(f"Sample folders found: {len(sample_dirs)}")
print(f"Coverage files read: {len(all_results)}")
print(f"Samples represented in merged workbook: {number_of_merged_samples}")
print(f"Configured TFBSs: {len(TARGET_SITES)}")
print(f"Missing coverage files: {len(missing_files)}")
print(f"Empty coverage files: {len(empty_files)}")
print(f"Failed files: {len(failed_files)}")
print(f"Invalid numeric rows removed: {invalid_numeric_rows}")
print(f"Duplicate sample-TFBS rows removed: {duplicate_count}")

if missing_sites:
    print(
        "TFBSs with no results: "
        + ", ".join(missing_sites)
    )
else:
    print(
        "All 44 configured TFBSs were detected."
    )

print(
    "\nUpdated workbook created:\n"
    f"{OUTPUT_XLSX}"
)

print(
    "\nModule update summary:"
)

print(
    pd.DataFrame(
        module_summary
    ).to_string(
        index=False
    )
)

Sample folders found: 769
Coverage files read: 769
Samples represented in merged workbook: 769
Configured TFBSs: 44
Missing coverage files: 0
Empty coverage files: 0
Failed files: 0
Invalid numeric rows removed: 0
Duplicate sample-TFBS rows removed: 0
All 44 configured TFBSs were detected.

Updated workbook created:
/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Griffin/griffin_nucleosome_profiling/Griffin_44TF_GC_corrected_summary_by_module.xlsx

Module update summary:
                 Module  Existing_TFBS  TFBS_added  Samples_added
             Interferon              3           0              0
         HLA_Regulation              5           0              0
         Breast_Lineage             11          14              0
           Bone_TGF_BMP              5           0              0
Osteoclast_Inflammation              3           0              0
          General_Tumor              2           0              0
      Technical_Control              1           0       